## Student setup

Run the modules in order: M1 creates the handoff dataset consumed by M2, M3, and M4. Keep the master dataset in approved private storage; it is intentionally excluded from this repository. Set `MEDML_MASTER_DATASET_PATH` for Module 1 and `MEDML_OUTPUT_DIR` if outputs should persist outside the repository.


# M4 | Hospitalization at ED triage

This notebook asks whether information available at ED triage can identify a stay with **hospitalization**. The target definition is explicit, the predictor boundary is enforced, and all performance results are recomputed locally.

## Clinical question and learning objectives

**Question:** Can a triage-time model rank ED stays by the likelihood of hospitalization?

Students will:

- reproduce and audit the target definition;
- separate target construction from predictors;
- quantify prevalence and the consequences of class imbalance;
- compare a dummy, logistic regression, random forest, and gradient boosting baseline;
- interpret confusion matrices, sensitivity, specificity, PPV, NPV, F1, ROC AUC, and average precision;
- inspect threshold, error, subgroup, calibration, and feature-influence behavior.

This is a retrospective benchmark exercise. A score is not a diagnosis or a deployment recommendation.

## Target definition: what counts as hospitalization?

`outcome_hospitalization` is true when `hadm_id` is present. Hospitalization is a disposition-related outcome observed after the triage decision; `hadm_id` is therefore used to audit the label and is strictly forbidden as an input feature. This module does not use in-hospital mortality as its target.

The target is a retrospective ground-truth-like label from the source table. It is not information that would be available at the moment of triage. We therefore verify it first and exclude every field used to construct it, plus downstream fields, from the feature matrix.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
import os

repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        candidate
        for candidate in repo_candidates
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir()
    ),
    Path.cwd(),
)
output_override = os.getenv("MEDML_OUTPUT_DIR")
OUTPUT_DIR = Path(output_override).expanduser() if output_override else REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = OUTPUT_DIR / "M1_dataset_for_next_module.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "M1 handoff not found. Run Module 1 first, or set MEDML_OUTPUT_DIR "
        "to the folder containing M1_dataset_for_next_module.csv."
    )
ROOT_DIR = REPO_ROOT
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded M1-ready dataset: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Hospitalization outcomes: {int(df['outcome_hospitalization'].sum()):,} ({df['outcome_hospitalization'].mean() * 100:.2f}%)")

In [ ]:
TARGET_COLUMN = "outcome_hospitalization"
MODULE_PREFIX = "M4"
hospitalization_from_hadm = df["hadm_id"].notna()
assert (hospitalization_from_hadm == df[TARGET_COLUMN]).all()
target_audit = pd.DataFrame({
    "definition_check": ["outcome equals hadm_id present", "positive stays", "negative stays", "prevalence_pct"],
    "value": [bool((hospitalization_from_hadm == df[TARGET_COLUMN]).all()), int(df[TARGET_COLUMN].sum()), int((~df[TARGET_COLUMN]).sum()), df[TARGET_COLUMN].mean() * 100],
})
display(target_audit.round(2))

## 1. Describe the target before modeling

Prevalence is the fraction of positive stays. It affects the prior probability of a positive prediction and therefore affects PPV and NPV. Report both counts and percentages, and inspect outcome prevalence across a small number of clinically legible groups before fitting anything.

In [ ]:
hospitalization_by_acuity = (
    df.groupby("triage_acuity", dropna=False)
    .agg(stays=("stay_id", "size"), hospitalized_count=(TARGET_COLUMN, "sum"), hospitalization_pct=(TARGET_COLUMN, "mean"))
    .reset_index()
)
hospitalization_by_acuity["hospitalization_pct"] *= 100
arrival_hospitalization = (
    df.groupby("arrival_transport", dropna=False)
    .agg(stays=("stay_id", "size"), hospitalization_pct=(TARGET_COLUMN, "mean"))
    .reset_index()
)
arrival_hospitalization["hospitalization_pct"] *= 100
display(hospitalization_by_acuity.round(2))
display(arrival_hospitalization.round(2))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=hospitalization_by_acuity, x="triage_acuity", y="hospitalization_pct", color="#0d9488", ax=axes[0])
axes[0].set(title="Hospitalization by triage acuity", xlabel="Triage acuity", ylabel="Hospitalization (%)")
sns.barplot(data=arrival_hospitalization, x="arrival_transport", y="hospitalization_pct", color="#ea580c", ax=axes[1])
axes[1].tick_params(axis="x", rotation=25)
axes[1].set(title="Hospitalization by arrival transport", xlabel="Arrival transport", ylabel="Hospitalization (%)")
plt.tight_layout()
plt.show()

## 2. Declare the triage-time predictor boundary

We use the benchmark feature family: demographics/context, prior ED/hospital/ICU utilization, triage measurements, chief-complaint flags, and CCI/ECI comorbidity indicators. Raw `chiefcomplaint` text is intentionally outside this tabular baseline.

Exclude identifiers, timestamps, disposition, ED departure information, ED LOS, hospitalization or critical labels, mortality/ICU timing, revisits, later ED measurements, and medication counts. A clinically plausible field is still leakage if it is recorded after the decision point.

In [ ]:
triage_features = [
    "age", "gender", "race", "arrival_transport", "triage_temperature",
    "triage_heartrate", "triage_resprate", "triage_o2sat", "triage_sbp",
    "triage_dbp", "triage_pain", "triage_acuity",
]
prior_features = [
    "n_ed_30d", "n_ed_90d", "n_ed_365d", "n_hosp_30d", "n_hosp_90d",
    "n_hosp_365d", "n_icu_30d", "n_icu_90d", "n_icu_365d",
]
complaint_features = [column for column in df.columns if column.startswith("chiefcom_")]
comorbidity_features = [
    column for column in df.columns
    if column.startswith("cci_") or column.startswith("eci_")
]
FEATURES = [
    column for column in triage_features + prior_features + complaint_features + comorbidity_features
    if column in df.columns
]
CATEGORICAL = [column for column in ["gender", "race", "arrival_transport"] if column in FEATURES]
NUMERIC = [column for column in FEATURES if column not in CATEGORICAL]
print(f"Predictors in the declared triage-time boundary: {len(FEATURES)}")
print("Categorical:", CATEGORICAL)
print("Numeric/binary:", len(NUMERIC))

In [ ]:
excluded_columns = [
    column for column in df.columns
    if column in {
        "index", "subject_id", "hadm_id", "stay_id", "intime", "outtime", "admittime",
        "dischtime", "deathtime", "edregtime", "edouttime", "disposition", "ed_los",
        "ed_los_hours", "outcome_inhospital_mortality", "outcome_icu_transfer_12h",
        "time_to_icu_transfer", "time_to_icu_transfer_hours", "outcome_hospitalization",
        "outcome_critical", "next_ed_visit_time", "next_ed_visit_time_diff",
        "next_ed_visit_time_diff_days", "outcome_ed_revisit_3d", "ed_temperature_last",
        "ed_heartrate_last", "ed_resprate_last", "ed_o2sat_last", "ed_sbp_last",
        "ed_dbp_last", "ed_pain_last", "n_med", "n_medrecon",
    }
]
print("Excluded fields represented in the source:")
print(excluded_columns)
assert not set(FEATURES) & set(excluded_columns)

## 3. Patient-disjoint split and training-only preprocessing

We use a reproducible 30,000-row teaching sample. A patient may have several ED stays, so `subject_id` is the grouping key. The test partition is held out while models are developed. Imputation, missingness indicators, scaling, and one-hot encoding live inside each pipeline and are fitted on training rows only.

In [ ]:
model_df = df.sample(n=min(50000, len(df)), random_state=42).reset_index(drop=True)
X = model_df[FEATURES]
y = model_df[TARGET_COLUMN].astype(int)
groups = model_df["subject_id"]
from sklearn.model_selection import GroupShuffleSplit
splitter = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
train_index, test_index = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_index], X.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]
groups_train, groups_test = groups.iloc[train_index], groups.iloc[test_index]
print(f"Train rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Train patients: {groups_train.nunique():,}; test patients: {groups_test.nunique():,}")
print("Patient overlap:", len(set(groups_train) & set(groups_test)))
print(f"Target prevalence: train={y_train.mean():.3f}; test={y_test.mean():.3f}")
assert set(groups_train).isdisjoint(set(groups_test))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_preprocessor():
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scaler", StandardScaler()),
                ]),
                NUMERIC,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", encoder),
                ]),
                CATEGORICAL,
            ),
        ],
        remainder="drop",
    )


def make_pipeline(estimator):
    return Pipeline([("preprocessor", make_preprocessor()), ("model", estimator)])

## 4. Candidate models and evaluation metrics

The prior dummy is the minimum baseline. Logistic regression is a transparent linear reference. Random forest can represent nonlinear interactions, while gradient boosting builds a sequence of small trees. The comparison is fair only when all models use the same rows, target, feature list, and held-out test partition.

At a threshold of 0.5, sensitivity is the fraction of positive outcomes found; specificity is the fraction of negatives left negative; PPV is the fraction of alerts that are positive; NPV is the fraction of negative predictions that are negative. ROC AUC and average precision summarize ranking across thresholds.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def classification_metrics(y_true, probabilities, threshold=0.5):
    predictions = np.asarray(probabilities) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    has_both_classes = len(np.unique(y_true)) == 2
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "sensitivity": recall_score(y_true, predictions, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "ppv": precision_score(y_true, predictions, zero_division=0),
        "npv": tn / (tn + fn) if tn + fn else np.nan,
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities) if has_both_classes else np.nan,
        "average_precision": average_precision_score(y_true, probabilities) if has_both_classes else np.nan,
        "true_positive": tp,
        "false_positive": fp,
        "true_negative": tn,
        "false_negative": fn,
    }

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "prior dummy": DummyClassifier(strategy="prior"),
    "logistic regression": make_pipeline(LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)),
    "random forest": make_pipeline(RandomForestClassifier(n_estimators=80, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=1, random_state=42)),
    "gradient boosting": make_pipeline(GradientBoostingClassifier(n_estimators=60, max_depth=2, learning_rate=.08, random_state=42)),
}
fitted_models = {}
probabilities = {}
result_rows = []
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    fitted_models[name] = estimator
    probabilities[name] = estimator.predict_proba(X_test)[:, 1]
    result_rows.append({"model": name, **classification_metrics(y_test, probabilities[name])})
model_results = pd.DataFrame(result_rows).sort_values("roc_auc", ascending=False)
display(model_results.round(3))

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

roc_curve_rows = []
pr_curve_rows = []
for model_name, model_probabilities in probabilities.items():
    false_positive_rate, true_positive_rate, _ = roc_curve(y_test, model_probabilities)
    precision, recall, _ = precision_recall_curve(y_test, model_probabilities)
    roc_curve_rows.extend(
        {"model": model_name, "false_positive_rate": fpr_value, "true_positive_rate": tpr_value}
        for fpr_value, tpr_value in zip(false_positive_rate, true_positive_rate)
    )
    pr_curve_rows.extend(
        {"model": model_name, "recall": recall_value, "precision": precision_value}
        for recall_value, precision_value in zip(recall, precision)
    )
M4_roc_curve_results = pd.DataFrame(roc_curve_rows)
M4_pr_curve_results = pd.DataFrame(pr_curve_rows)
M4_roc_curve_results.to_csv(OUTPUT_DIR / "M4_roc_curve_results.csv", index=False)
M4_pr_curve_results.to_csv(OUTPUT_DIR / "M4_pr_curve_results.csv", index=False)
print(f"Wrote {len(M4_roc_curve_results):,} ROC points and {len(M4_pr_curve_results):,} PR points")

## 5. ROC and precision-recall views

ROC AUC is useful for ranking but can look reassuring when the number of false alerts is operationally large. Precision-recall curves make the positive class and its prevalence visible. Read the curves together with the confusion matrix at the threshold a real user would actually apply.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, model_probabilities in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, model_probabilities)
    precision, recall, _ = precision_recall_curve(y_test, model_probabilities)
    axes[0].plot(fpr, tpr, label=f"{name}: {roc_auc_score(y_test, model_probabilities):.3f}")
    axes[1].plot(recall, precision, label=f"{name}: AP={average_precision_score(y_test, model_probabilities):.3f}")
axes[0].plot([0, 1], [0, 1], ":", color="black")
axes[1].axhline(y_test.mean(), linestyle=":", color="black", label=f"prevalence={y_test.mean():.3f}")
axes[0].set(title="ROC comparison", xlabel="False-positive rate", ylabel="Sensitivity")
axes[1].set(title="Precision-recall comparison", xlabel="Sensitivity / precision", ylabel="Precision / PPV")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Threshold analysis and confusion matrix

A threshold is a policy choice. Lowering it can capture more positive outcomes while increasing false positives; raising it can reduce alerts while missing more positives. Use the table to connect a model score to a named action. Do not call 0.5 “the clinical threshold” without a decision specification.

In [ ]:
selected_model_name = "gradient boosting"
selected_probabilities = probabilities[selected_model_name]
threshold_results = pd.DataFrame([
    {"model": selected_model_name, **classification_metrics(y_test, selected_probabilities, threshold)}
    for threshold in [.10, .25, .40, .50, .60, .75]
])
display(threshold_results.round(3))
selected_threshold = .50
selected_predictions = selected_probabilities >= selected_threshold
confusion = pd.DataFrame(
    confusion_matrix(y_test, selected_predictions, labels=[0, 1]),
    index=["observed 0", "observed 1"],
    columns=["predicted 0", "predicted 1"],
)
display(confusion)

## 7. Calibration: does a probability mean what it says?

Discrimination asks whether higher scores tend to belong to positive cases. Calibration asks whether groups assigned a probability of, for example, 0.70 experience the outcome about 70% of the time. A model can rank cases well and still be poorly calibrated. The Brier score is the mean squared error of the predicted probabilities; lower is better on the same test population.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

fraction_positive, mean_predicted = calibration_curve(
    y_test, selected_probabilities, n_bins=10, strategy="quantile"
)
calibration_results = pd.DataFrame({
    "mean_predicted_probability": mean_predicted,
    "observed_fraction_positive": fraction_positive,
})
print(f"Brier score: {brier_score_loss(y_test, selected_probabilities):.4f}")
display(calibration_results.round(3))
plt.figure(figsize=(6, 6))
plt.plot(mean_predicted, fraction_positive, "o-", color="#ea580c", label="model")
plt.plot([0, 1], [0, 1], ":", color="black", label="perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction positive")
plt.title(f"Calibration for {selected_model_name}")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Cross-validation inside the training partition

    The test partition is not used for model selection. Grouped three-fold cross-validation inside training gives a sense of variation when patients are held apart in each fold. It is not an external validation study and cannot repair target bias, measurement bias, or a non-transportable workflow.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score
cv = GroupKFold(n_splits=3)
cv_pipeline = make_pipeline(LogisticRegression(max_iter=500, class_weight="balanced", random_state=42))
cv_scores = cross_val_score(cv_pipeline, X_train, y_train, groups=groups_train, cv=cv, scoring="roc_auc", n_jobs=1)
cv_summary = pd.DataFrame({"fold": np.arange(1, len(cv_scores) + 1), "roc_auc": cv_scores})
display(cv_summary.round(3))
print(f"Grouped CV ROC AUC mean={cv_scores.mean():.3f}; SD={cv_scores.std(ddof=1):.3f}")

## 9. Global feature influence: useful but bounded

Permutation importance shuffles one raw input in held-out rows and measures the resulting performance change through the complete pipeline. It answers “how much does this input help this model on these rows?” It does not answer “what causes the outcome?” Correlated variables can share or hide importance, and workflow variables may not transport.

In [ ]:
from sklearn.inspection import permutation_importance
importance_sample = X_test.sample(n=min(5000, len(X_test)), random_state=42)
importance_labels = y_test.loc[importance_sample.index]
importance = permutation_importance(
    fitted_models[selected_model_name], importance_sample, importance_labels,
    scoring="roc_auc", n_repeats=3, random_state=42, n_jobs=1,
)
importance_table = (
    pd.DataFrame({"feature": importance_sample.columns, "mean_auc_drop": importance.importances_mean, "sd_auc_drop": importance.importances_std})
    .sort_values("mean_auc_drop", ascending=False)
)
display(importance_table.head(20).round(4))

## 10. Explainability for the M4 Gradient Boosting model

This section explains only the selected M4 Gradient Boosting model on held-out test rows. Native tree importance measures how often transformed predictors contribute to the fitted trees; SHAP values attribute each prediction to feature contributions relative to the model's expected output. Mean absolute SHAP describes contribution magnitude, while signed SHAP describes direction. These are model explanations, not causal effects, and correlated or one-hot encoded predictors should be interpreted as related feature groups.

In [ ]:
import shap

gb_pipeline = fitted_models["gradient boosting"]
gb_model = gb_pipeline.named_steps["model"]
gb_preprocessor = gb_pipeline.named_steps["preprocessor"]
explainability_sample = X_test.sample(n=min(2000, len(X_test)), random_state=42)
explainability_labels = y_test.loc[explainability_sample.index]
transformed_sample = gb_preprocessor.transform(explainability_sample)
transformed_feature_names = list(gb_preprocessor.get_feature_names_out())
assert transformed_sample.shape[1] == len(transformed_feature_names) == len(gb_model.feature_importances_)

def source_feature_name(transformed_name):
    name = transformed_name.split("__", 1)[-1]
    for feature in sorted(FEATURES, key=len, reverse=True):
        if (
            name == feature
            or name.startswith(f"{feature}_")
            or name == f"missingindicator_{feature}"
            or name.startswith(f"missingindicator_{feature}_")
        ):
            return feature
    return name

source_features = [source_feature_name(name) for name in transformed_feature_names]

native_importance = pd.DataFrame({
    "transformed_feature": transformed_feature_names,
    "source_feature": source_features,
    "gb_feature_importance": gb_model.feature_importances_,
})
gb_importance_table = (
    native_importance.groupby("source_feature", as_index=False)["gb_feature_importance"]
    .sum()
    .sort_values(by="gb_feature_importance", ascending=False)
)

explainer = shap.TreeExplainer(gb_model)
raw_shap_values = explainer.shap_values(transformed_sample)
if isinstance(raw_shap_values, list):
    shap_values = np.asarray(raw_shap_values[1])
else:
    shap_values = np.asarray(raw_shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]
assert shap_values.shape == transformed_sample.shape

shap_values_transformed = pd.DataFrame(
    shap_values, index=explainability_sample.index, columns=transformed_feature_names
)
shap_values_sample = shap_values_transformed.copy()
shap_values_sample.insert(0, "row_index", shap_values_sample.index)
shap_values_sample = shap_values_sample.reset_index(drop=True)

shap_values_by_source = pd.DataFrame(index=shap_values_transformed.index)
for source_feature in dict.fromkeys(source_features):
    source_columns = [
        transformed_feature
        for transformed_feature, mapped_source in zip(transformed_feature_names, source_features)
        if mapped_source == source_feature
    ]
    shap_values_by_source[source_feature] = shap_values_transformed[source_columns].sum(axis=1)

shap_feature_summary = pd.DataFrame({
    "source_feature": shap_values_by_source.columns,
    "mean_abs_shap": shap_values_by_source.abs().mean().values,
    "mean_shap": shap_values_by_source.mean().values,
    "positive_fraction": (shap_values_by_source > 0).mean().values,
    "min_shap": shap_values_by_source.min().values,
    "max_shap": shap_values_by_source.max().values,
}).sort_values(by="mean_abs_shap", ascending=False)

row_explanation_rows = []
for row_index, contributions in shap_values_by_source.iterrows():
    positive = contributions.nlargest(3)
    negative = contributions.nsmallest(3)
    row_explanation_rows.append({
        "row_index": row_index,
        "observed_outcome": int(explainability_labels.loc[row_index]),
        "predicted_probability": gb_pipeline.predict_proba(explainability_sample.loc[[row_index]])[0, 1],
        "largest_positive_contributions": "; ".join(f"{feature} ({value:+.3f})" for feature, value in positive.items()),
        "largest_negative_contributions": "; ".join(f"{feature} ({value:+.3f})" for feature, value in negative.items()),
    })
shap_row_explanations = pd.DataFrame(row_explanation_rows)

display(native_importance.sort_values(by="gb_feature_importance", ascending=False).head(20).round(4))
display(gb_importance_table.head(20).round(4))
display(shap_feature_summary.head(20).round(4))
display(shap_row_explanations.head(10))

In [ ]:
top_native = gb_importance_table.head(15).sort_values(by="gb_feature_importance")
top_shap = shap_feature_summary.head(15).sort_values(by="mean_abs_shap")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=top_native, x="gb_feature_importance", y="source_feature", color="#0d9488", ax=axes[0])
axes[0].set(title="Native Gradient Boosting importance", xlabel="Summed split importance", ylabel="Source feature")
sns.barplot(data=top_shap, x="mean_abs_shap", y="source_feature", color="#ea580c", ax=axes[1])
axes[1].set(title="Mean absolute SHAP value", xlabel="Average contribution magnitude", ylabel="Source feature")
plt.tight_layout()
plt.show()

### How to read these M4 explanations

- **Native GB importance:** triage acuity contributes the largest share of tree split importance, followed by age, arrival transport, and prior hospitalization history. This describes how the fitted trees partition the transformed data; it is not a direction of risk.
- **SHAP magnitude:** the same predictors have the largest average absolute contributions. A larger value means a larger typical movement away from the model's baseline output for a held-out row.
- **SHAP direction:** positive values push the model toward hospitalization and negative values push it away. The row-level table shows the strongest positive and negative source-feature contributions for individual test rows. These values are on the estimator's model-output scale, not direct percentage-point changes in probability.
- **Cautions:** importance is not causality. Correlated predictors can divide credit, one-hot levels are summarized to their source feature, and these explanations describe this M4 dataset, split, preprocessing, and fitted model rather than guaranteed clinical behavior.

## 11. Subgroup performance and limitations

Overall performance can conceal different behavior across gender, race, or arrival transport. Report subgroup size and prevalence next to sensitivity, specificity, PPV, NPV, and AUC. Small groups and intersectional groups need uncertainty intervals in a full study; a descriptive table is not a complete fairness analysis.

In [ ]:
test_view = X_test[[column for column in ["gender", "race", "arrival_transport"] if column in X_test.columns]].copy()
test_view["observed"] = y_test.to_numpy()
test_view["probability"] = selected_probabilities
subgroup_rows = []
for group_column in ["gender", "race", "arrival_transport"]:
    for group_value, group in test_view.groupby(group_column, dropna=False):
        if len(group) < 50:
            continue
        subgroup_rows.append({
            "group_variable": group_column,
            "group": str(group_value),
            "n": len(group),
            "prevalence": group["observed"].mean(),
            **{key: value for key, value in classification_metrics(group["observed"], group["probability"]).items() if key in {"sensitivity", "specificity", "ppv", "npv", "roc_auc"}},
        })
subgroup_results = pd.DataFrame(subgroup_rows)
display(subgroup_results.round(3))

## M4 handoff and final questions

The notebook exports computed results, not copied historical claims. Ask:

1. What information is available at triage and what is downstream?
2. Which error matters most for the intended action, and why?
3. How does prevalence change PPV and NPV?
4. What does feature importance fail to establish?
5. What validation, calibration, fairness, and workflow evidence is missing?

In [ ]:
M3_or_M4_tables = {
    "model_results.csv": model_results,
    "threshold_results.csv": threshold_results,
    "calibration_results.csv": calibration_results,
    "cv_results.csv": cv_summary,
    "feature_importance.csv": importance_table,
    "subgroup_results.csv": subgroup_results,
}
for filename, table in M3_or_M4_tables.items():
    table.to_csv(OUTPUT_DIR / f"{MODULE_PREFIX}_{filename}", index=False)
print(f"Wrote {len(M3_or_M4_tables)} {MODULE_PREFIX} tables to {OUTPUT_DIR}")

In [ ]:
M4_explainability_tables = {
    "gradient_boosting_importance.csv": gb_importance_table,
    "shap_feature_summary.csv": shap_feature_summary,
    "shap_values_sample.csv": shap_values_sample,
    "shap_row_explanations.csv": shap_row_explanations,
}
for filename, table in M4_explainability_tables.items():
    table.to_csv(OUTPUT_DIR / f"{MODULE_PREFIX}_{filename}", index=False)
print(f"Wrote {len(M4_explainability_tables)} M4 explainability tables to {OUTPUT_DIR}")